# 10.5 — Move controlled data

Delete the existing `DatasetForPublish/raw_data` and `controlled_data` trees (if exist), copy `time_shifted` into a new `raw_data` tree, move Driver Video, Eyeball video, and complete raw ET recording directories to `controlled_data`, and delete IDUN PDF reports. Relative paths are preserved and empty source directories are removed.

In [1]:
from pathlib import Path
import sys
import shutil

START = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (START,) + tuple(START.parents)
    if (path / 'ImBrAid' / 'config.json').is_file()
)
sys.path.insert(0, str(PROJECT_ROOT))
from helpers.project_config import load_project_config

CONFIG = load_project_config(PROJECT_ROOT / 'ImBrAid' / 'config.json')
PROCESSING_DATA_ROOT = CONFIG.path('paths', 'processing_data_root')
RAW_DATA_ROOT = CONFIG.path('paths', 'publication_raw_data_root')
CONTROLLED_DATA_ROOT = CONFIG.path('paths', 'not_public_controled_data_root')

In [2]:
RUN = True
driver_subfolder = CONFIG.value('modalities', 'driver_video_source_subfolder')
et_subfolder = CONFIG.value('modalities', 'eyeball_source_subfolder')

In [3]:
moved = {
    'Driver videos': 0,
    'Eyeball videos': 0,
    'Raw ET recording directories': 0,
}
deleted_idun_pdfs = 0
removed_empty_dirs = 0

if RUN:
    for target_root in (RAW_DATA_ROOT, CONTROLLED_DATA_ROOT):
        if target_root.exists():
            shutil.rmtree(target_root)
    shutil.copytree(PROCESSING_DATA_ROOT, RAW_DATA_ROOT)

    participant_dirs = sorted(
        (path for path in RAW_DATA_ROOT.iterdir() if path.is_dir()),
        key=lambda path: path.name.casefold(),
    )
    for participant_dir in participant_dirs:
        driver_dir = participant_dir / driver_subfolder
        et_dir = participant_dir / et_subfolder
        idun_dir = participant_dir / 'IDUN'

        driver_videos = [] if not driver_dir.is_dir() else [
            path for path in driver_dir.iterdir()
            if path.is_file() and path.suffix.casefold() == '.mkv'
        ]
        et_paths = [] if not et_dir.is_dir() else list(et_dir.iterdir())
        eyeball_videos = [
            path for path in et_paths
            if path.is_file() and path.name.casefold().endswith('_eyeball_video.webm')
        ]
        raw_et_dirs = [
            path for path in et_paths
            if path.is_dir() and (
                path.name.casefold().endswith('_et_raw')
                or '_et_raw_' in path.name.casefold()
            )
        ]
        idun_pdfs = [] if not idun_dir.is_dir() else [
            path for path in idun_dir.iterdir()
            if path.is_file() and path.suffix.casefold() == '.pdf'
        ]

        move_groups = {
            'Driver videos': driver_videos,
            'Eyeball videos': eyeball_videos,
            'Raw ET recording directories': raw_et_dirs,
        }
        for name, paths in move_groups.items():
            for source in paths:
                target = CONTROLLED_DATA_ROOT / source.relative_to(RAW_DATA_ROOT)
                target.parent.mkdir(parents=True, exist_ok=True)
                source.rename(target)
                moved[name] += 1

        for source in idun_pdfs:
            source.unlink()
            deleted_idun_pdfs += 1

        for path in (driver_dir, et_dir, idun_dir):
            if path.is_dir() and not any(path.iterdir()):
                path.rmdir()
                removed_empty_dirs += 1

{'copied_records': len(participant_dirs), 'moved': moved,
 'deleted_idun_pdfs': deleted_idun_pdfs,
 'removed_empty_dirs': removed_empty_dirs}

{'copied_records': 43,
 'moved': {'Driver videos': 43,
  'Eyeball videos': 36,
  'Raw ET recording directories': 44},
 'deleted_idun_pdfs': 26,
 'removed_empty_dirs': 86}